# Week 12c: Reasoning and Planning Agents
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement the ReAct (Reason + Act) pattern** — Interleave explicit reasoning traces with tool use and observations.
2. **Build multi-step tool-chaining agents** — Compose tools in sequence where each step consumes intermediate results.
3. **Design self-reflection and self-correction loops** — Generate, evaluate, and revise outputs until a quality bar is met.
4. **Implement error recovery and replanning** — Detect tool failures and switch strategies instead of stopping blindly.
5. **Understand memory management in reasoning agents** — Use scratchpads for short-term chains and retrieval for longer-lived knowledge, and know when to persist vs forget.

---
## Setup — Install & Imports

Install core libraries for **LangGraph**, **LangChain**, and model backends. If you do not have an OpenAI key, this notebook falls back to **Flan-T5** for text generation (smaller models follow instructions less reliably—treat outputs as *pedagogical demos*).

In [ ]:
!pip install -q langgraph langchain langchain-openai openai transformers

In [ ]:
import os
import re
import json
import math
import random
from getpass import getpass
from typing import TypedDict, Literal, Annotated, Optional
from dataclasses import dataclass, field
from collections import Counter
from operator import add

if not os.environ.get("OPENAI_API_KEY"):
    try:
        key = getpass("Enter your OpenAI API key (or press Enter to skip): ")
        if key.strip():
            os.environ["OPENAI_API_KEY"] = key.strip()
    except Exception:
        pass

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OpenAI API key configured: {HAS_OPENAI}")
if not HAS_OPENAI:
    print("→ Using Flan-T5 (google/flan-t5-base) as fallback. ReAct parsing may need more steps or manual checks.")

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage

if HAS_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    def chat(messages: list) -> str:
        resp = llm.invoke(messages)
        return resp.content if hasattr(resp, "content") else str(resp)
else:
    from transformers import pipeline

    _pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_length=512)

    def chat(messages: list) -> str:
        parts = []
        for m in messages:
            role = getattr(m, "type", None) or m.__class__.__name__.replace("Message", "").lower()
            content = m.content if hasattr(m, "content") else str(m)
            parts.append(f"{role}: {content}")
        prompt = "\n".join(parts)
        out = _pipe(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"]
        return out.strip()

print("LLM backend ready.")

### Running this notebook

- **Local / Jupyter:** Run cells top to bottom. The first time you hit the OpenAI path, `getpass` prompts in the terminal or notebook input area.
- **Colab:** Store `OPENAI_API_KEY` in *Secrets* or paste when prompted; otherwise the Flan-T5 path downloads weights on first use.
- **Cost:** This notebook uses `gpt-4o-mini` when a key is present; swap models if you are running many automated tests.


---
## 1. The ReAct Pattern

**ReAct** ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) interleaves **reasoning** and **acting**:

- **Thought** — The model explains what it knows and what it intends to do next.
- **Action** — A structured call to an external **tool** (calculator, API, database, …).
- **Observation** — The environment returns a result; the model continues reasoning.

This explicit trace makes debugging and supervision easier than a single opaque answer, because you can inspect *why* each tool was chosen.

### Tools and a minimal ReAct loop (from scratch)

We define three tools—**calculator**, **string manipulator**, and **knowledge lookup**—then run a small **parser + executor** loop. The model must emit lines our parser understands (`Thought:`, `Action:`, `Action Input:`). If it emits `Final Answer:`, we stop.

### Example ReAct trace (idealized)

```text
Thought: I need the population of France before scaling.
Action: knowledge_lookup
Action Input: population of France 2023 estimate
Observation: about 68.4 million
Thought: Convert to millions with arithmetic.
Action: calculator
Action Input: round(68.4)
Observation: 68
Final Answer: POP_MILLIONS=68; ...
```

Live traces differ by model; the **parser** enforces structure.


In [ ]:
def tool_calculator(expr: str) -> str:
    """Safely evaluate arithmetic expressions with math helpers only."""
    allowed = {"__builtins__": {}, "math": math, "abs": abs, "min": min, "max": max, "round": round}
    try:
        return str(eval(expr, allowed, {}))
    except Exception as e:
        return f"Error: {e}"


def tool_string_manipulator(cmd: str) -> str:
    """cmd format: OP|arg1|arg2...  OP in UPPER, REVERSE, COUNT_WORDS"""
    parts = [p.strip() for p in cmd.split("|")]
    if not parts:
        return "Error: empty command"
    op = parts[0].upper()
    if op == "UPPER" and len(parts) >= 2:
        return parts[1].upper()
    if op == "REVERSE" and len(parts) >= 2:
        return parts[1][::-1]
    if op == "COUNT_WORDS" and len(parts) >= 2:
        return str(len(parts[1].split()))
    return "Error: unknown string op or missing args"


KNOWLEDGE_BASE = {
    "capital of france": "Paris",
    "population of france 2023 estimate": "about 68.4 million",
    "gdp of france 2023 usd": "about 2.78e12",
}


def tool_knowledge_lookup(query: str) -> str:
    q = query.strip().lower()
    for k, v in KNOWLEDGE_BASE.items():
        if k in q or q in k:
            return v
    # fuzzy: overlap
    qtok = set(q.split())
    best, score = None, 0
    for k, v in KNOWLEDGE_BASE.items():
        kt = set(k.split())
        s = len(qtok & kt)
        if s > score:
            best, score = v, s
    return best if score else "No matching fact in toy knowledge base."


TOOLS = {
    "calculator": tool_calculator,
    "string_manipulator": tool_string_manipulator,
    "knowledge_lookup": tool_knowledge_lookup,
}

TOOL_DESC = """
You have tools:
- calculator: input is a Python arithmetic expression using + - * / ** and math functions (e.g. "2**10", "math.sqrt(2)").
- string_manipulator: input like UPPER|hello or REVERSE|abc or COUNT_WORDS|one two three.
- knowledge_lookup: input is a short natural-language question; returns a string fact from a small KB.
""".strip()

In [ ]:
def parse_react_block(text: str) -> dict:
    """Extract Thought, Action, Action Input, or Final Answer from model output."""
    out = {"thought": None, "action": None, "action_input": None, "final": None}
    fa = re.search(r"Final Answer:\s*(.+)", text, re.I | re.S)
    if fa:
        out["final"] = fa.group(1).strip()
        return out
    th = re.search(r"Thought:\s*(.+?)(?=\n(?:Action:|Final Answer:)|\Z)", text, re.I | re.S)
    ac = re.search(r"Action:\s*(\w+)", text, re.I)
    ai = re.search(r"Action Input:\s*(.+?)(?=\n(?:Thought:|Observation:|Final Answer:)|\Z)", text, re.I | re.S)
    if th:
        out["thought"] = th.group(1).strip()
    if ac:
        out["action"] = ac.group(1).strip().lower()
    if ai:
        out["action_input"] = ai.group(1).strip()
    return out


def run_react_agent(question: str, max_steps: int = 8) -> None:
    scratchpad = []
    system = (
        "You are a ReAct agent. Always use this exact format each turn:\n"
        "Thought: ...\n"
        "Action: tool_name\n"
        "Action Input: ...\n"
        "When done, output exactly one line:\n"
        "Final Answer: ...\n\n"
        f"{TOOL_DESC}\n"
        "Valid Action names: calculator, string_manipulator, knowledge_lookup."
    )
    if not HAS_OPENAI:
        system += "\nKeep answers short. Follow the format strictly."

    for step in range(max_steps):
        obs_block = "\n".join(scratchpad)
        user = f"Question: {question}\n\n{obs_block}\nWhat is your next Thought and Action?"
        messages = [SystemMessage(content=system), HumanMessage(content=user)]
        reply = chat(messages)
        print(f"\n--- Step {step + 1} (raw model) ---\n{reply}\n")
        parsed = parse_react_block(reply)
        if parsed.get("final"):
            print("✓ Final Answer:", parsed["final"])
            return
        act, a_in = parsed.get("action"), parsed.get("action_input")
        if not act or a_in is None:
            scratchpad.append(f"Observation: Could not parse Action/Action Input. Reply again with the required format.")
            continue
        fn = TOOLS.get(act)
        if not fn:
            scratchpad.append(f"Observation: Unknown action '{act}'. Use calculator, string_manipulator, or knowledge_lookup.")
            continue
        obs = fn(a_in)
        scratchpad.append(f"Observation: {obs}")
        print(f"→ Parsed Action: {act}({a_in!r}) → Observation: {obs}")

    print("Stopped: max_steps reached without Final Answer.")


In [ ]:
# Multi-step problem: combine KB + calculator + string manipulator
q = (
    "Look up the population of France (estimate). Then compute how many people that is if we round to the nearest million "
    "(use calculator: divide by 1e6, then round). Finally, use string_manipulator to UPPER the word 'react'."
    " Concatenate in Final Answer as: POP_MILLIONS=<value>; TAG=<uppercase react>."
)
run_react_agent(q, max_steps=10)

---
## 2. Multi-Step Tool Chaining (LangGraph)

Some workflows are **inherently sequential**: later tools need **numeric or structured outputs** from earlier tools. Here we model:

1. **Lookup** population of France (toy KB).
2. **Convert** population to **millions**.
3. **Compute** per-capita GDP given a fixed GDP figure from the KB.

LangGraph’s `StateGraph` makes the **dataflow** explicit: each node reads/writes shared state, unlike an opaque chain of string prompts.

In [ ]:
def parse_population_to_float(text: str) -> float:
    m = re.search(r"([\d.]+)\s*million", text, re.I)
    if m:
        return float(m.group(1)) * 1e6
    m = re.search(r"([\d.]+)\s*billion", text, re.I)
    if m:
        return float(m.group(1)) * 1e9
    m = re.search(r"([\d.]+)", text.replace(",", ""))
    if m:
        return float(m.group(1)) * 1e6 if float(m.group(1)) < 1000 else float(m.group(1))
    raise ValueError(f"Could not parse population from: {text!r}")


def parse_gdp_to_float(text: str) -> float:
    m = re.search(r"([\d.eE+-]+)", text.replace(",", ""))
    if not m:
        raise ValueError(text)
    return float(m.group(1))


class ChainState(TypedDict):
    query: str
    population_raw: str
    population: float
    population_millions: float
    gdp_usd: float
    gdp_per_capita: float
    trace: Annotated[list, add]


def node_lookup_population(state: ChainState):
    raw = tool_knowledge_lookup("population of France 2023 estimate")
    tr = {"step": "lookup_population", "input": "population of France", "output": raw}
    return {"population_raw": raw, "trace": [tr]}


def node_to_millions(state: ChainState):
    pop = parse_population_to_float(state["population_raw"])
    millions = round(pop / 1e6, 2)
    tr = {"step": "to_millions", "input": state["population_raw"], "output": millions}
    return {"population": pop, "population_millions": millions, "trace": [tr]}


def node_gdp_per_capita(state: ChainState):
    gdp_s = tool_knowledge_lookup("gdp of france 2023 usd")
    gdp = parse_gdp_to_float(gdp_s)
    pc = gdp / state["population"]
    tr = {"step": "gdp_per_capita", "input": {"gdp": gdp, "population": state["population"]}, "output": pc}
    return {"gdp_usd": gdp, "gdp_per_capita": pc, "trace": [tr]}


chain_graph = StateGraph(ChainState)
chain_graph.add_node("lookup_population", node_lookup_population)
chain_graph.add_node("to_millions", node_to_millions)
chain_graph.add_node("gdp_per_capita", node_gdp_per_capita)
chain_graph.set_entry_point("lookup_population")
chain_graph.add_edge("lookup_population", "to_millions")
chain_graph.add_edge("to_millions", "gdp_per_capita")
chain_graph.add_edge("gdp_per_capita", END)
chain_app = chain_graph.compile()

In [ ]:
init: ChainState = {
    "query": "France population → millions → per capita GDP",
    "population_raw": "",
    "population": 0.0,
    "population_millions": 0.0,
    "gdp_usd": 0.0,
    "gdp_per_capita": 0.0,
    "trace": [],
}
result = chain_app.invoke(init)
print("Population (raw):", result["population_raw"])
print("Population (parsed):", result["population"])
print("Population (millions):", result["population_millions"])
print("GDP (USD):", result["gdp_usd"])
print("GDP per capita (USD):", round(result["gdp_per_capita"], 2))
print("\nTool chain trace:")
for row in result["trace"]:
    print(json.dumps(row, indent=2))

---
## 3. Self-Reflection and Self-Correction

A **reflective** agent:

1. **Drafts** an answer.
2. **Critiques** it (rubric, checklist, or numeric score).
3. **Revises** if the critique fails a threshold.

We implement a tiny LangGraph with nodes `draft → reflect → (revise loop or end)`. The **critic** returns JSON: `{"score": 1-10, "pass": bool, "feedback": "..."}`. We log scores per iteration to **measure improvement**.

In [ ]:
class ReflectState(TypedDict):
    question: str
    draft: str
    reflection: str
    score: float
    passed: bool
    iteration: int
    max_iters: int
    history: Annotated[list, add]


def extract_json_obj(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            return json.loads(m.group(0))
        raise


def node_draft(state: ReflectState):
    it = state["iteration"]
    extra = ""
    if it > 0 and state.get("reflection"):
        extra = f"\nPrior critique (fix these issues): {state['reflection']}"
    prompt = (
        f"Answer concisely with one short paragraph. Question: {state['question']}{extra}\n"
        "Requirements: mention 'France', give a population order-of-magnitude, and state GDP per capita is rough/back-of-envelope."
    )
    draft = chat([HumanMessage(content=prompt)])
    return {"draft": draft, "history": [{"iter": it, "event": "draft", "text": draft[:400]}]}


def node_reflect(state: ReflectState):
    crit_prompt = f"""You grade student answers. Return ONLY valid JSON with keys score (number 1-10), pass (boolean), feedback (string).
Pass if score >= 8.
Question: {state['question']}
Answer: {state['draft']}
Check: mentions France; population scale; acknowledges estimate/uncertainty."""
    raw = chat([HumanMessage(content=crit_prompt)])
    try:
        obj = extract_json_obj(raw)
        score = float(obj.get("score", 0))
        passed = bool(obj.get("pass", score >= 8))
        fb = str(obj.get("feedback", ""))
    except Exception:
        score, passed, fb = 5.0, False, "Critic JSON parse failed; revise for clarity and constraints."
    refl = json.dumps({"score": score, "pass": passed, "feedback": fb})
    return {
        "reflection": fb,
        "score": score,
        "passed": passed,
        "history": [{"iter": state["iteration"], "event": "reflect", "score": score, "pass": passed}],
    }


def node_bump_iter(state: ReflectState):
    return {"iteration": state["iteration"] + 1}


def route_after_reflect(state: ReflectState) -> Literal["bump", "end"]:
    if state["passed"] or state["iteration"] + 1 >= state["max_iters"]:
        return "end"
    return "bump"


rg = StateGraph(ReflectState)
rg.add_node("draft", node_draft)
rg.add_node("reflect", node_reflect)
rg.add_node("bump", node_bump_iter)
rg.set_entry_point("draft")
rg.add_edge("draft", "reflect")
rg.add_conditional_edges("reflect", route_after_reflect, {"bump": "bump", "end": END})
rg.add_edge("bump", "draft")
reflect_app = rg.compile()

In [ ]:
rq = "Summarize France's economic scale using population and GDP per capita (rough estimates)."
r0 = reflect_app.invoke(
    {
        "question": rq,
        "draft": "",
        "reflection": "",
        "score": 0.0,
        "passed": False,
        "iteration": 0,
        "max_iters": 3,
        "history": [],
    }
)
scores = [h["score"] for h in r0["history"] if h.get("event") == "reflect"]
print("Final draft:\n", r0["draft"])
print("\nReflection scores over iterations:", scores)
if len(scores) >= 2:
    print("Δ (last - first):", round(scores[-1] - scores[0], 2))
print("\nFull history (truncated):")
for h in r0["history"]:
    print(h)

### Chaining as a graph

```mermaid
flowchart LR
  A[lookup_population] --> B[to_millions]
  B --> C[gdp_per_capita]
```

Each node reads **prior state** produced upstream—deterministic control flow with explicit data dependencies.


---
## 4. Error Recovery and Replanning

Real agents hit **timeouts**, **schema errors**, and **unexpected observations**. A robust pattern is:

1. **Detect** failure (exception, sentinel string, validation).
2. **Replan** — try an alternate tool or decompose the task.
3. **Escalate** — ask a human when automated recovery is exhausted.

Below: **Tool A** (flaky) → on failure **Tool B** → then **human ask** (here we simulate with a placeholder string rather than blocking on `input()` in batch runs).

In [ ]:
class RecoveryState(TypedDict):
    x: float
    y: float
    result: Optional[float]
    log: Annotated[list, add]
    status: Literal["ok", "need_user"]


def flaky_divide(a: float, b: float, fail_prob: float = 0.7) -> float:
    if random.random() < fail_prob:
        raise TimeoutError("Tool A: simulated timeout")
    return a / b


def safe_divide(a: float, b: float) -> float:
    if b == 0:
        raise ValueError("division by zero")
    return a / b


def node_try_a(state: RecoveryState):
    log = []
    try:
        r = flaky_divide(state["x"], state["y"], fail_prob=0.65)
        log.append({"tool": "A", "ok": True, "value": r})
        return {"result": r, "log": log, "status": "ok"}
    except Exception as e:
        log.append({"tool": "A", "ok": False, "error": str(e)})
        return {"log": log}


def route_a(state: RecoveryState) -> Literal["b", "done"]:
    return "done" if state.get("result") is not None else "b"


def node_try_b(state: RecoveryState):
    log = []
    try:
        r = safe_divide(state["x"], state["y"])
        log.append({"tool": "B", "ok": True, "value": r})
        return {"result": r, "log": log, "status": "ok"}
    except Exception as e:
        log.append({"tool": "B", "ok": False, "error": str(e)})
        return {"log": log}


def route_b(state: RecoveryState) -> Literal["human", "done"]:
    return "done" if state.get("result") is not None else "human"


def node_ask_user(state: RecoveryState):
    msg = "Tools A and B failed; escalate to user with the inputs x,y and error summary."
    log = [{"tool": "human", "ok": False, "prompt": msg, "x": state["x"], "y": state["y"]}]
    return {"status": "need_user", "log": log}


rec = StateGraph(RecoveryState)
rec.add_node("try_a", node_try_a)
rec.add_node("try_b", node_try_b)
rec.add_node("ask_user", node_ask_user)
rec.set_entry_point("try_a")
rec.add_conditional_edges("try_a", route_a, {"b": "try_b", "done": END})
rec.add_conditional_edges("try_b", route_b, {"human": "ask_user", "done": END})
rec.add_edge("ask_user", END)
recovery_app = rec.compile()

In [ ]:
random.seed(42)
for trial in range(4):
    out = recovery_app.invoke({"x": 22.0, "y": 7.0, "result": None, "log": [], "status": "ok"})
    print(f"\n--- Trial {trial + 1} ---")
    print("status:", out["status"], "result:", out.get("result"))
    for row in out["log"]:
        print(" ", row)

---
## 5. Memory Management: Scratchpad vs Long-Term Store

| Memory | Role | Typical lifetime |
|--------|------|------------------|
| **Short-term (scratchpad)** | Holds the *current* reasoning trace, tool I/O, partial plans | Cleared each task or each session |
| **Long-term (retrieval store)** | Reuses *past facts*, user prefs, or distilled lessons across tasks | Persisted in a DB / vector index |

**When to persist:** stable user settings, verified facts, successful plan templates. **When to forget:** stale web results, failed attempts that might bias the model, PII you should not retain.

Below we implement a **scratchpad** (plain list) and a **tiny long-term memory** using lexical overlap (no extra packages). Production systems usually swap this retriever for embeddings + a vector DB.

In [ ]:
def tokenize(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", s.lower())


def lexical_score(query: str, doc: str) -> float:
    q, d = Counter(tokenize(query)), Counter(tokenize(doc))
    if not q:
        return 0.0
    dot = sum(q[t] * d.get(t, 0) for t in q)
    norm = math.sqrt(sum(v * v for v in q.values())) * math.sqrt(sum(v * v for v in d.values()))
    return dot / norm if norm else 0.0


@dataclass
class AgentMemory:
    scratchpad: list[str] = field(default_factory=list)
    long_term: list[tuple[str, str]] = field(default_factory=list)  # (title, content)

    def remember_fact(self, title: str, content: str) -> None:
        self.long_term.append((title.strip(), content.strip()))

    def recall(self, query: str, k: int = 2) -> list[tuple[str, str, float]]:
        scored = [(t, c, lexical_score(query, t + " " + c)) for t, c in self.long_term]
        scored.sort(key=lambda x: x[2], reverse=True)
        return scored[:k]

    def think(self, line: str) -> None:
        self.scratchpad.append(line)

    def clear_scratchpad(self) -> None:
        self.scratchpad.clear()

In [ ]:
mem = AgentMemory()
mem.remember_fact("User preference", "User prefers answers in USD and short bullet summaries.")
mem.remember_fact("Project Acme", "Acme deadline is October 15; primary contact is team-ml@example.com.")
mem.remember_fact("France fact", "France population is on the order of 68M; GDP per capita is mid‑tens of k USD (estimate).")

task = "Draft a one-sentence note about France's population scale for the Acme project update."
mem.clear_scratchpad()
mem.think(f"Task: {task}")
retrieved = mem.recall(task, k=2)
mem.think("Retrieved long-term snippets: " + "; ".join(f"{t} ({s:.2f})" for t, _, s in retrieved))
ctx = "\n".join(f"- {t}: {c}" for t, c, _ in retrieved)
prompt = f"Use the context. {task}\nContext:\n{ctx}"
answer = chat([HumanMessage(content=prompt)])
mem.think("Draft: " + answer)

print("=== Scratchpad (ephemeral chain-of-thought + IO) ===")
for line in mem.scratchpad:
    print("-", line[:500])
print("\n=== Long-term store (durable facts) ===")
for t, c in mem.long_term:
    print(f"- {t}: {c}")

---
## Exercises

1. **Domain ReAct agent:** Pick a domain (biology, finance, sports analytics, …). Implement **three or more** custom tools and a ReAct loop that solves a **multi-hop** question in your domain. Include at least one tool that returns structured data parsed by a later step.
2. **Measure reflection:** Take any generator from this notebook (ReAct final answer, chain summary, or `draft` node). Add a **reflection** step with a fixed rubric. Plot or tabulate **scores across iterations** for 5 different questions; report when reflection helps vs hurts.
3. **Replanning depth:** Implement a policy that tries **at least two distinct approaches** (different tools or decompositions) before returning “I give up.” Log each attempt and the failure reason. Bonus: add a cost model (e.g., fake USD per tool call) and minimize expected cost subject to a success rate target.

### Designing the critic

Reflection quality depends on the **rubric**. If the critic only scores verbosity, models can game the metric while staying factually weak. Prefer **checkable** criteria (required entities, numeric sanity bands, explicit uncertainty) and log failure modes for debugging.


---
## Responsible AI Reflection

Reasoning agents can plan and execute multi-step strategies autonomously. This power comes with risk: an agent that can reason about how to achieve a goal can also reason about how to circumvent safety guardrails. As reasoning capabilities improve, how do we ensure that agent planning remains aligned with human intentions? Is it enough to constrain the tools an agent can use, or do we need to constrain the reasoning process itself?

**Discussion prompts:** Tool allowlists reduce *capability* but not always *intent*; chain-of-thought monitoring can leak sensitive data yet help auditing; human checkpoints trade latency for control. Where would you draw the line in a high-stakes deployment (health, finance, physical systems)?

---
## Summary & Next Steps

- **ReAct** makes reasoning and tool use **inspectable**; robust parsing and clear tool schemas matter as much as model choice.
- **LangGraph** expresses **sequential and conditional** tool flows with explicit **state**, which helps testing and error handling.
- **Self-reflection** turns a one-shot generator into a **small optimization loop**—define pass/fail criteria carefully to avoid infinite churn or reward hacking.
- **Recovery graphs** encode **fallbacks** and **escalation** instead of failing silently.
- **Memory:** use a **scratchpad** for the current episode and **persistent retrieval** only for vetted, policy-compliant knowledge.

**Next steps:** Pair this notebook with **Week 12b (Deep Research)** for retrieval-heavy planning, and **Week 12a (Multi-Agent)** for role-specialized reasoning. Consider implementing a single **evaluation harness** (trajectory logging, success criteria, cost) shared across all three.